<a href="https://colab.research.google.com/github/ashwiniv07/A-Synergistic-Deep-Fusion-and-Adversarial-Defense-Framework-/blob/main/ProtocolB_IEEE_CIS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# CELL 1 — mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
BASE = '/content/drive/MyDrive/second paper '
f = BASE + '/protocol_b.py'
s = open(f).read()
old = """    X = tf.constant(S['Xte'][idx, None, :], tf.float32); y = tf.constant(S['yte'][idx], tf.int32)
    ce = keras.losses.SparseCategoricalCrossentropy()
    with tf.GradientTape() as tape:
        tape.watch(X); loss = ce(y, net(X, training=False)[0])
    Xa = X + CFG['eps_train'] * tf.sign(tape.gradient(loss, X))
    dc = net(X, training=False)[1].numpy().ravel(); da = net(Xa, training=False)[1].numpy().ravel()"""
new = """    ce = keras.losses.SparseCategoricalCrossentropy()
    dcs, das = [], []
    for i in range(0, len(idx), 1024):
        X = tf.constant(S['Xte'][idx[i:i + 1024], None, :], tf.float32)
        y = tf.constant(S['yte'][idx[i:i + 1024]], tf.int32)
        with tf.GradientTape() as tape:
            tape.watch(X); loss = ce(y, net(X, training=False)[0])
        Xa = X + CFG['eps_train'] * tf.sign(tape.gradient(loss, X))
        dcs.append(net(X, training=False)[1].numpy().ravel()); das.append(net(Xa, training=False)[1].numpy().ravel())
    dc = np.concatenate(dcs); da = np.concatenate(das)"""
print('found' if old in s else 'ALREADY CHANGED or NOT FOUND')
s = s.replace(old, new)
open(f, 'w').write(s)

found


41764

In [ ]:
# CELL 2 — install libraries
!pip -q install lightgbm xgboost imbalanced-learn statsmodels

In [ ]:
# CELL 3 — setup
import warnings; warnings.filterwarnings('ignore')
import sys, importlib
BASE = '/content/drive/MyDrive/second paper '
DATA_DIR = BASE + '/data'
sys.path.insert(0, BASE)
import protocol_b as pb
importlib.reload(pb)

SMOKE = False
pb.init(BASE + '/protocol_b_results/ieee', dataset='ieee', smoke=SMOKE,
        seeds_main=[0, 1, 2], pool_legit_cap=50000, max_epochs=100)
import tensorflow as tf
print('GPU:', tf.config.list_physical_devices('GPU'))

Results -> /content/drive/MyDrive/second paper /protocol_b_results/ieee | dataset ieee 
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [ ]:
# CELL 4 — load and split the data (chronological 70/10/20)
S = pb.chrono_split(pb.load_ieee(DATA_DIR + '/ieee-fraud-detection'))
pb.register(S)

IEEE-CIS: (590540, 51) fraud rate 0.0350 | D = 51
  tr: n=413378  fraud=14538 (0.0352)
  va: n=59053  fraud=2061 (0.0349)
  te: n=118109  fraud=4064 (0.0344)


In [ ]:
# CELL 5 — Table 7 (main comparison) + timings for Table 11
pb.run_all(S, paysim=False, tables=(7,))


=== Table 7: main comparison ===
  lgbm                   seed 0  PR-AUC 0.2871  R@1%FPR 0.2195
  training cnn / neurosync / seed 0 on 29076 rows (14538 fraud)
    epoch 1  val PR-AUC 0.1033  (best 0.1033)  5s
    epoch 2  val PR-AUC 0.1317  (best 0.1317)  7s
    epoch 3  val PR-AUC 0.1382  (best 0.1382)  7s
    epoch 4  val PR-AUC 0.1460  (best 0.1460)  2s
    epoch 5  val PR-AUC 0.1521  (best 0.1521)  7s
    epoch 6  val PR-AUC 0.1576  (best 0.1576)  2s
    epoch 7  val PR-AUC 0.1604  (best 0.1604)  3s
    epoch 8  val PR-AUC 0.1601  (best 0.1604)  2s
    epoch 9  val PR-AUC 0.1635  (best 0.1635)  3s
    epoch 10  val PR-AUC 0.1621  (best 0.1635)  3s
    epoch 11  val PR-AUC 0.1646  (best 0.1646)  2s
    epoch 12  val PR-AUC 0.1654  (best 0.1654)  3s
    epoch 13  val PR-AUC 0.1654  (best 0.1654)  3s
    epoch 14  val PR-AUC 0.1664  (best 0.1664)  2s
    epoch 15  val PR-AUC 0.1685  (best 0.1685)  2s
    epoch 16  val PR-AUC 0.1694  (best 0.1694)  3s
    epoch 17  val PR-AUC 0.1688 

In [ ]:
# CELL 6 — Table 8 (imbalance treatments)
pb.run_all(S, paysim=False, tables=(8,))


=== Table 8: imbalance treatments ===

=== statistics ===
      baseline      mcnemar_p      delong_p  mcnemar_p_holm  delong_p_holm
0           lr   0.000000e+00  9.343015e-07    0.000000e+00   9.343015e-07
1          xgb   1.245885e-46  1.376802e-25    2.491769e-46   5.507208e-25
2         lgbm  1.789174e-101  6.525338e-49   7.156698e-101   3.262669e-48
3          cnn  1.081216e-144  1.882465e-07   5.406079e-144   3.764930e-07
4        bigru   2.483540e-24  3.482264e-14    2.483540e-24   1.044679e-13
5  transformer   1.586486e-86  1.034496e-59    4.759457e-86   6.206979e-59
done in 0.2 min


In [ ]:
# CELL 7 — Table 9 (ablation)
pb.run_all(S, paysim=False, tables=(9,))


=== Table 9: ablation ===
  training fgsm_no_disc / neurosync / seed 1 on 29076 rows (14538 fraud)
    epoch 1  val PR-AUC 0.1172  (best 0.1172)  21s
    epoch 2  val PR-AUC 0.1246  (best 0.1246)  5s
    epoch 3  val PR-AUC 0.1248  (best 0.1248)  9s
    epoch 4  val PR-AUC 0.1368  (best 0.1368)  6s
    epoch 5  val PR-AUC 0.1458  (best 0.1458)  5s
    epoch 6  val PR-AUC 0.1476  (best 0.1476)  6s
    epoch 7  val PR-AUC 0.1520  (best 0.1520)  5s
    epoch 8  val PR-AUC 0.1573  (best 0.1573)  6s
    epoch 9  val PR-AUC 0.1582  (best 0.1582)  6s
    epoch 10  val PR-AUC 0.1624  (best 0.1624)  6s
    epoch 11  val PR-AUC 0.1649  (best 0.1649)  6s
    epoch 12  val PR-AUC 0.1680  (best 0.1680)  6s
    epoch 13  val PR-AUC 0.1697  (best 0.1697)  6s
    epoch 14  val PR-AUC 0.1718  (best 0.1718)  6s
    epoch 15  val PR-AUC 0.1626  (best 0.1718)  5s
    epoch 16  val PR-AUC 0.1696  (best 0.1718)  6s
    epoch 17  val PR-AUC 0.1685  (best 0.1718)  5s
    epoch 18  val PR-AUC 0.1696  (best 0.

In [ ]:
f = BASE + '/protocol_b.py'
s = open(f).read()
s = s.replace("def run_discriminator(S, seed, n=20000):", "def run_discriminator(S, seed, n=4000):")
open(f, 'w').write(s)
importlib.reload(pb)
pb.init(BASE + '/protocol_b_results/ieee', dataset='ieee', smoke=False,
        seeds_main=[0, 1, 2], pool_legit_cap=50000, max_epochs=100)
pb.register(S)
print('fixed')

Results -> /content/drive/MyDrive/second paper /protocol_b_results/ieee | dataset ieee 
fixed


In [ ]:
# CELL 8 — Table 10 (attacks) + discriminator
pb.run_all(S, paysim=False, tables=(10,))


=== Table 10: attacks ===
  attacks cnn seed 1: {'clean': 0.2894, 'fgsm_0.01': 0.2655, 'fgsm_0.05': 0.1636, 'fgsm_0.1': 0.0674, 'pgd_0.05': 0.1592, 'cpgd_0.1': 0.2795}
  attacks no_adv seed 1: {'clean': 0.2212, 'fgsm_0.01': 0.1924, 'fgsm_0.05': 0.091, 'fgsm_0.1': 0.0317, 'pgd_0.05': 0.0837, 'cpgd_0.1': 0.2013}
  attacks proposed seed 1: {'clean': 0.3189, 'fgsm_0.01': 0.2936, 'fgsm_0.05': 0.2109, 'fgsm_0.1': 0.0965, 'pgd_0.05': 0.2067, 'cpgd_0.1': 0.2987}
  attacks cnn seed 2: {'clean': 0.2862, 'fgsm_0.01': 0.2645, 'fgsm_0.05': 0.171, 'fgsm_0.1': 0.0716, 'pgd_0.05': 0.1673, 'cpgd_0.1': 0.267}
  attacks no_adv seed 2: {'clean': 0.2571, 'fgsm_0.01': 0.2259, 'fgsm_0.05': 0.123, 'fgsm_0.1': 0.0416, 'pgd_0.05': 0.1134, 'cpgd_0.1': 0.2328}
  attacks proposed seed 2: {'clean': 0.2485, 'fgsm_0.01': 0.2205, 'fgsm_0.05': 0.1366, 'fgsm_0.1': 0.0632, 'pgd_0.05': 0.1289, 'cpgd_0.1': 0.2283}

=== statistics ===
      baseline      mcnemar_p      delong_p  mcnemar_p_holm  delong_p_holm
0           lr

In [ ]:
# CELL 9 — final tables (saved as tables_ieee.xlsx in second paper /protocol_b_results/ieee)
T = pb.summary(paysim=False)
for name, df in T.items():
    print('\n' + name); display(df)

Saved /content/drive/MyDrive/second paper /protocol_b_results/ieee/tables_ieee.xlsx

t7


,Model,PR-AUC,ROC-AUC,Recall@1% FPR,Recall@0.1% FPR,Precision,Recall,F1,MCC
0,Logistic regression,0.1390 ± 0.0000,0.7727 ± 0.0000,0.0967 ± 0.0000,0.0084 ± 0.0000,0.1460 ± 0.0000,0.4400 ± 0.0000,0.2193 ± 0.0000,0.2083 ± 0.0000
1,XGBoost,0.2634 ± 0.0055,0.8183 ± 0.0055,0.2087 ± 0.0071,0.0600 ± 0.0067,0.3268 ± 0.0172,0.3137 ± 0.0255,0.3194 ± 0.0109,0.2961 ± 0.0106
2,LightGBM,0.2901 ± 0.0026,0.8367 ± 0.0010,0.2266 ± 0.0062,0.0700 ± 0.0046,0.3497 ± 0.0068,0.3246 ± 0.0013,0.3366 ± 0.0027,0.3142 ± 0.0032
3,CNN,0.1617 ± 0.0079,0.7871 ± 0.0021,0.1270 ± 0.0106,0.0171 ± 0.0032,0.2084 ± 0.0101,0.3007 ± 0.0224,0.2456 ± 0.0033,0.2179 ± 0.0039
4,BiGRU,0.1376 ± 0.0146,0.7524 ± 0.0166,0.1086 ± 0.0074,0.0139 ± 0.0081,0.1869 ± 0.0349,0.2872 ± 0.0100,0.2250 ± 0.0263,0.1964 ± 0.0262
5,Transformer,0.1224 ± 0.0007,0.7249 ± 0.0032,0.0824 ± 0.0036,0.0085 ± 0.0017,0.1806 ± 0.0048,0.2881 ± 0.0075,0.2219 ± 0.0038,0.1932 ± 0.0040
6,Proposed (full),0.1536 ± 0.0057,0.7837 ± 0.0015,0.1160 ± 0.0064,0.0210 ± 0.0046,0.1951 ± 0.0243,0.2747 ± 0.0385,0.2262 ± 0.0131,0.1978 ± 0.0142



t8


,Imbalance treatment,PR-AUC,Recall@1% FPR,Sampling time (s)
0,Class-weighted loss,0.1662 ± 0.0301,0.1263 ± 0.0256,0 ± 0
1,Random undersampling,0.1333 ± 0.0076,0.0977 ± 0.0038,0 ± 0
2,SMOTE,0.1561 ± 0.0185,0.1137 ± 0.0150,2 ± 1
3,"NeuroSync (full fitness, Eq. 5)",0.1536 ± 0.0057,0.1160 ± 0.0064,3 ± 0
4,"NeuroSync (density only, γ = δ = 0)",0.0936 ± 0.0008,0.0399 ± 0.0019,3 ± 1



t9


,Variant,PR-AUC
0,Full model,0.1536 ± 0.0057
1,Without CNN branch,0.1502 ± 0.0087
2,Without BiGRU branch,0.1678 ± 0.0026
3,Without Transformer branch,0.1560 ± 0.0049
4,Attention fusion replaced by concatenation,0.1654 ± 0.0120
5,Attention fusion replaced by averaging of bran...,0.1663 ± 0.0051
6,Ensemble of the three separately trained branc...,0.1682 ± 0.0062
7,"Without adversarial training (γ = 0, no discri...",0.1430 ± 0.0074
8,FGSM training without the GRL discriminator,0.1536 ± 0.0058
9,"Full model, columns randomly permuted",0.1572 ± 0.0122


... (2 outputs left out)


t11


,Model,Trainable parameters,Training time per epoch (s),"Inference latency, batch 1 (ms)","Throughput, batch 1,024 (transactions/s)"
0,LightGBM,–,–,1.40,2905
1,CNN,12642,3.3,0.78,66358
2,BiGRU,36130,33.5,5.19,5984
3,Transformer,48226,61.9,2.14,2111
4,Proposed,138274,6.1,4.96,52427



disc


,disc_accuracy,disc_auc,disc_bce,mean_clean,mean_adv
0,0.5126 ± 0.0029,0.5143 ± 0.0021,0.6927 ± 0.0000,0.5022 ± 0.0030,0.5013 ± 0.0030
